In [2]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
from src.kite_fetcher import _get_kite

## 1. Margins

In [3]:
kite = _get_kite()

margins = kite.margins()

equity_margin = margins.get('equity', {})
commodity_margin = margins.get('commodity', {})

margin_summary = pd.DataFrame([
    {
        'segment': 'equity',
        'available_cash':    equity_margin.get('available', {}).get('cash', 0),
        'available_intraday': equity_margin.get('available', {}).get('intraday_payin', 0),
        'used_debits':       equity_margin.get('utilised', {}).get('debits', 0),
        'used_exposure':     equity_margin.get('utilised', {}).get('exposure', 0),
        'net':               equity_margin.get('net', 0),
    },
    {
        'segment': 'commodity',
        'available_cash':    commodity_margin.get('available', {}).get('cash', 0),
        'available_intraday': commodity_margin.get('available', {}).get('intraday_payin', 0),
        'used_debits':       commodity_margin.get('utilised', {}).get('debits', 0),
        'used_exposure':     commodity_margin.get('utilised', {}).get('exposure', 0),
        'net':               commodity_margin.get('net', 0),
    }
])

margin_summary

,segment,available_cash,available_intraday,used_debits,used_exposure,net
0,equity,0,0,2.911167e+06,654670.6975,807450.16748
1,commodity,0,0,0.000000e+00,0.0000,0.00000


## 2. Open Positions

In [4]:
positions_raw = kite.positions()

# 'net' = net positions across day + overnight
# 'day' = intraday positions only
net_positions = pd.DataFrame(positions_raw.get('net', []))
day_positions = pd.DataFrame(positions_raw.get('day', []))

if not net_positions.empty:
    cols = ['tradingsymbol', 'exchange', 'quantity', 'average_price',
            'last_price', 'pnl', 'value', 'buy_quantity', 'sell_quantity']
    cols = [c for c in cols if c in net_positions.columns]
    net_positions = net_positions[cols]

print(f"Net positions: {len(net_positions)}")
net_positions

Net positions: 40


,tradingsymbol,exchange,quantity,average_price,last_price,pnl,value,buy_quantity,sell_quantity
0,GOLDM26MAR150000PE,MCX,-1,2675.00,1035.00,16400.00,-26750.00,0,1
1,GOLDM26MAR160000PE,MCX,-1,2400.00,5562.50,-31625.00,-24000.00,0,1
2,GOLDM26MAR163000CE,MCX,-1,1013.00,933.50,795.00,-10130.00,0,1
3,GOLDM26MAR165000CE,MCX,-1,712.00,667.50,445.00,-7120.00,0,1
4,SILVERM26MAR260000PE,MCX,-1,8370.00,11000.00,-13150.00,-41850.00,0,1
5,SILVERM26MAR280000CE,MCX,-1,4860.50,5240.00,-1897.50,-24302.50,0,1
6,BAJFINANCE26MAR840PE,NFO,750,13.60,11.10,-1875.00,10200.00,750,0
7,BAJFINANCE26MAR860PE,NFO,750,19.90,17.05,-2137.50,14925.00,750,0
8,BAJFINANCE26MAR900CE,NFO,-750,19.10,11.90,5400.00,-14325.00,0,750
9,BAJFINANCE26MAR910PE,NFO,-1500,45.90,44.20,2550.00,-68850.00,0,1500


In [ ]:
# Day (intraday) positions
print(f"Day positions: {len(day_positions)}")
day_positions[['tradingsymbol', 'exchange', 'quantity', 'average_price', 'last_price', 'pnl']] \
    if not day_positions.empty else print("No intraday positions")

## 3. Holdings (Delivery / Long-term)

In [ ]:
holdings_raw = kite.holdings()
holdings = pd.DataFrame(holdings_raw)

if not holdings.empty:
    cols = ['tradingsymbol', 'exchange', 'quantity', 'average_price',
            'last_price', 'pnl', 'day_change', 'day_change_percentage']
    cols = [c for c in cols if c in holdings.columns]
    holdings = holdings[cols]
    holdings['market_value'] = holdings['quantity'] * holdings['last_price']

print(f"Holdings: {len(holdings)}")
holdings

## 4. Portfolio P&L Summary

In [ ]:
if not holdings.empty:
    total_invested = (holdings['average_price'] * holdings['quantity']).sum()
    total_market_value = holdings['market_value'].sum()
    total_pnl = holdings['pnl'].sum()
    total_return_pct = (total_pnl / total_invested) * 100 if total_invested else 0

    summary = pd.DataFrame([{
        'total_invested':    round(total_invested, 2),
        'market_value':      round(total_market_value, 2),
        'total_pnl':         round(total_pnl, 2),
        'return_%':          round(total_return_pct, 2),
    }])
    display(summary)

    # P&L by stock
    holdings.set_index('tradingsymbol')['pnl'].sort_values().plot(
        kind='barh', title='P&L by Stock', figsize=(10, 5), color='steelblue'
    )
    plt.xlabel('P&L (₹)')
    plt.tight_layout()
    plt.show()

## 5. Portfolio Allocation

In [ ]:
if not holdings.empty:
    holdings.set_index('tradingsymbol')['market_value'].plot(
        kind='pie', title='Portfolio Allocation by Market Value',
        figsize=(7, 7), autopct='%1.1f%%'
    )
    plt.ylabel('')
    plt.tight_layout()
    plt.show()